Aquí tienes el script completo y riguroso para implementar la **Recomendación 2 (Métodos de Regularización)**.

### Consideraciones estadísticas del diseño:

1. **La naturaleza de LASSO vs Elastic Net**: Como estos algoritmos aplican penalizaciones lineales directamente sobre las características exógenas, **no se utiliza el motor de SARIMAX (que asume optimización por máxima verosimilitud ordinaria)**. En su lugar, se emplean `LassoCV` y `ElasticNetCV` de la librería `scikit-learn` con validación cruzada integrada para encontrar de forma automática los coeficientes óptimos ($\alpha$ y la relación L1/L2) que minimizan el sobreajuste.
2. **Inclusión de la Dinámica Temporal**: Para competir de forma justa con un modelo de series temporales, las matrices de características ($X$) incluyen tanto los rezagos de las variables meteorológicas como los **rezagos autorregresivos directos de los casos de dengue**.
3. **Escalamiento**: Se aplica un escalamiento estricto sobre el set de entrenamiento y se transfiere al de testeo para evitar fugas de información (*data leakage*).

```python


In [1]:
# =============================================================================
# PASO 1: IMPORTACIÓN DE LIBRERÍAS DE ALTA PRECISIÓN
# =============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

sns.set_theme(style="whitegrid")

# =============================================================================
# PASO 2: CONFIGURACIÓN DE RUTAS Y CARGA DE DATOS (2023 - 2025)
# =============================================================================
ruta_datos = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\6_recomendacion_estadistica_2\2_datos\1_raw\2_meteo_epi_rezagos_meteo_epi.xlsx"
dir_resultados = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\6_recomendacion_estadistica_2\3_resultados"

os.makedirs(dir_resultados, exist_ok=True)

print(f"[INFO] Cargando espacio muestral de alta dimensionalidad desde:\n{ruta_datos}")
df = pd.read_excel(ruta_datos)

# Configurar índice cronológico
df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')
df.set_index('fecha', inplace=True)
df = df.asfreq('W')
df = df.ffill().bfill()

# Restringir temporalidad conforme a los experimentos previos
print("[INFO] Filtrando datos para el periodo estable: 2023 - 2025.")
df = df.loc['2023-01-01':'2025-12-31']

# Extraer el rango de años dinámicamente para los títulos
anio_min, anio_max = df.index.year.min(), df.index.year.max()
periodo_str = f"{anio_min}-{anio_max}"

# Agregar dinámicamente los rezagos de la variable objetivo si no existen en el archivo original
if 'casos_dengue_lag_1' not in df.columns:
    df['casos_dengue_lag_1'] = df['casos_dengue'].shift(1)
    df['casos_dengue_lag_2'] = df['casos_dengue'].shift(2)

df = df.dropna()

# =============================================================================
# PASO 3: SEPARACIÓN DE VARIABLES
# =============================================================================
y = df['casos_dengue']

columnas_exclusoras = ['casos_dengue', 'año', 'semana_epi', 'casos_ln']
columnas_exogenas = [col for col in df.columns if col not in columnas_exclusoras]
X_features = df[columnas_exogenas]

print(f"[INFO] Dataset listo para regularización. Total de predictores en la matriz X: {X_features.shape[1]}")

# =============================================================================
# PASO 4: REJILLA DE PARTICIONES CRONOLÓGICAS (95%, 96%, 97%)
# =============================================================================
particiones = {
    "95-5":  0.95,
    "96-4":  0.96,
    "97-3":  0.97
}

resultados_globales = []

# Iterar sobre las particiones para entrenar LASSO y ElasticNet de forma simultánea
for nombre_split, tasa_train in particiones.items():
    print("\n" + "="*80)
    print(f" PROCESANDO VENTANA TEMPORAL CRONOLÓGICA: {nombre_split}")
    print("="*80)
    
    # 1. Split de Datos
    tamanio_train = int(len(df) * tasa_train)
    y_train, y_test = y.iloc[:tamanio_train], y.iloc[tamanio_train:]
    X_train, X_test = X_features.iloc[:tamanio_train], X_features.iloc[tamanio_train:]
    
    # 2. Escalamiento (Requisito fundamental para regularización L1 y L2)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 3. IMPLEMENTACIÓN LASSO CON VALIDACIÓN CRUZADA (L1)
    print("[INFO] Ajustando selector adaptativo LASSO CV...")
    model_lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
    model_lasso.fit(X_train_scaled, y_train)
    
    y_pred_train_lasso = model_lasso.predict(X_train_scaled)
    y_pred_test_lasso = model_lasso.predict(X_test_scaled)
    
    mae_train_lasso = mean_absolute_error(y_train, y_pred_train_lasso)
    mae_test_lasso = mean_absolute_error(y_test, y_pred_test_lasso)
    features_activas_lasso = np.sum(model_lasso.coef_ != 0)
    
    # 4. IMPLEMENTACIÓN ELASTIC NET CON VALIDACIÓN CRUZADA (L1 + L2)
    print("[INFO] Ajustando selector adaptativo Elastic Net CV...")
    model_enet = ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1], cv=5, random_state=42, max_iter=10000)
    model_enet.fit(X_train_scaled, y_train)
    
    y_pred_train_enet = model_enet.predict(X_train_scaled)
    y_pred_test_enet = model_enet.predict(X_test_scaled)
    
    mae_train_enet = mean_absolute_error(y_train, y_pred_train_enet)
    mae_test_enet = mean_absolute_error(y_test, y_pred_test_enet)
    features_activas_enet = np.sum(model_enet.coef_ != 0)
    
    # Almacenar métricas en el reporte unificado
    resultados_globales.append({
        "Partición": nombre_split, "Algoritmo": "LASSO",
        "MAE Train": mae_train_lasso, "MAE Test": mae_test_lasso, "Variables Retenidas": features_activas_lasso
    })
    resultados_globales.append({
        "Partición": nombre_split, "Algoritmo": "Elastic Net",
        "MAE Train": mae_train_enet, "MAE Test": mae_test_enet, "Variables Retenidas": features_activas_enet
    })
    
    # =========================================================================
    # PASO 5: GRAFICACIÓN INDEPENDIENTE POR PARTICIÓN (COMPARATIVA DE DESEMPEÑO)
    # =========================================================================
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 10), sharey=True)
    
    # Gráficos de LASSO
    axes[0, 0].plot(y_train.index, y_train.values, label='Real Train', color='#1f77b4', alpha=0.7)
    axes[0, 0].plot(y_train.index, y_pred_train_lasso, label='LASSO Pred', color='#ff7f0e', linestyle='--')
    axes[0, 0].set_title(f"LASSO - Ajuste Train {nombre_split} (MAE: {mae_train_lasso:.4f})")
    axes[0, 0].legend()
    
    axes[0, 1].plot(y_test.index, y_test.values, label='Real Test', color='#2ca02c', alpha=0.7)
    axes[0, 1].plot(y_test.index, y_pred_test_lasso, label='LASSO Forecast', color='#d62728', linestyle='--')
    axes[0, 1].set_title(f"LASSO - Pronóstico Test {nombre_split} (MAE: {mae_test_lasso:.4f})")
    axes[0, 1].legend()
    
    # Gráficos de Elastic Net
    axes[1, 0].plot(y_train.index, y_train.values, label='Real Train', color='#1f77b4', alpha=0.7)
    axes[1, 0].plot(y_train.index, y_pred_train_enet, label='ElasticNet Pred', color='#9467bd', linestyle='--')
    axes[1, 0].set_title(f"Elastic Net - Ajuste Train {nombre_split} (MAE: {mae_train_enet:.4f})")
    axes[1, 0].legend()
    
    axes[1, 1].plot(y_test.index, y_test.values, label='Real Test', color='#2ca02c', alpha=0.7)
    axes[1, 1].plot(y_test.index, y_pred_test_enet, label='ElasticNet Forecast', color='#8c564b', linestyle='--')
    axes[1, 1].set_title(f"Elastic Net - Pronóstico Test {nombre_split} (MAE: {mae_test_enet:.4f})")
    axes[1, 1].legend()
    
    plt.suptitle(f"Comparativa de Métodos Estadísticos de Regularización | Partición {nombre_split} ({periodo_str})", 
                 fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    
    # Salida con etiquetas dinámicas de periodo
    ruta_grafico = os.path.join(dir_resultados, f"comparativa_regularizacion_split_{nombre_split}_{periodo_str}.png")
    plt.savefig(ruta_grafico, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[INFO] Imagen comparativa guardada con éxito para la partición {nombre_split}.")

# =============================================================================
# PASO 6: CONSOLIDACIÓN TABULAR Y REPORTE CIENTÍFICO FINAL
# =============================================================================
df_reporte = pd.DataFrame(resultados_globales)

# Añadir filas promedio calculadas por algoritmo para simplificar la toma de decisiones metodológicas
promedios = []
for algo, sub_df in df_reporte.groupby("Algoritmo"):
    promedios.append(pd.DataFrame([{
        "Partición": "PROMEDIO", "Algoritmo": algo,
        "MAE Train": sub_df["MAE Train"].mean(), "MAE Test": sub_df["MAE Test"].mean(),
        "Variables Retenidas": int(round(sub_df["Variables Retenidas"].mean()))
    }]))

df_reporte_completo = pd.concat([df_reporte, pd.concat(promedios)], ignore_index=True)

# Imprimir reporte en consola
print("\n" + "="*95)
print(f"      REPORTE DE DESEMPEÑO: CONTRACCIÓN DE PARÁMETROS (REGULARIZACIÓN LINEAL {periodo_str})      ")
print("="*95)
print(df_reporte_completo.to_string(index=False, formatters={
    "MAE Train": "{:.4f}".format, "MAE Test": "{:.4f}".format, "Variables Retenidas": "{:d}".format
}))
print("="*95)

# Guardar base en Excel
ruta_excel = os.path.join(dir_resultados, f"desempeno_regularizacion_{periodo_str}.xlsx")
df_reporte_completo.to_excel(ruta_excel, index=False)
print(f"\n[ÉXITO] Archivo de datos históricos consolidado en Excel:\n{ruta_excel}")


[INFO] Cargando espacio muestral de alta dimensionalidad desde:
C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\6_recomendacion_estadistica_2\2_datos\1_raw\2_meteo_epi_rezagos_meteo_epi.xlsx
[INFO] Filtrando datos para el periodo estable: 2023 - 2025.
[INFO] Dataset listo para regularización. Total de predictores en la matriz X: 155

 PROCESANDO VENTANA TEMPORAL CRONOLÓGICA: 95-5
[INFO] Ajustando selector adaptativo LASSO CV...


c:\Users\marco\Documentos\investigacion\arima\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.569e+00, tolerance: 6.890e+00
  model = cd_fast.enet_coordinate_descent(


[INFO] Ajustando selector adaptativo Elastic Net CV...


c:\Users\marco\Documentos\investigacion\arima\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.569e+00, tolerance: 6.890e+00
  model = cd_fast.enet_coordinate_descent(


[INFO] Imagen comparativa guardada con éxito para la partición 95-5.

 PROCESANDO VENTANA TEMPORAL CRONOLÓGICA: 96-4
[INFO] Ajustando selector adaptativo LASSO CV...


...

[INFO] Imagen comparativa guardada con éxito para la partición 96-4.

 PROCESANDO VENTANA TEMPORAL CRONOLÓGICA: 97-3
[INFO] Ajustando selector adaptativo LASSO CV...


c:\Users\marco\Documentos\investigacion\arima\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.727e+00, tolerance: 7.019e+00
  model = cd_fast.enet_coordinate_descent(
c:\Users\marco\Documentos\investigacion\arima\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.223e+00, tolerance: 7.019e+00
  model = cd_fast.enet_coordinate_descent(


[INFO] Ajustando selector adaptativo Elastic Net CV...


c:\Users\marco\Documentos\investigacion\arima\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.727e+00, tolerance: 7.019e+00
  model = cd_fast.enet_coordinate_descent(
c:\Users\marco\Documentos\investigacion\arima\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.223e+00, tolerance: 7.019e+00
  model = cd_fast.enet_coordinate_descent(


[INFO] Imagen comparativa guardada con éxito para la partición 97-3.

      REPORTE DE DESEMPEÑO: CONTRACCIÓN DE PARÁMETROS (REGULARIZACIÓN LINEAL 2023-2025)      
Partición   Algoritmo MAE Train MAE Test Variables Retenidas
     95-5       LASSO    7.4961   7.3205                   9
     95-5 Elastic Net    7.4961   7.3205                   9
     96-4       LASSO    7.4587   8.1782                   9
     96-4 Elastic Net    7.4587   8.1782                   9
     97-3       LASSO    7.3029   9.0863                  11
     97-3 Elastic Net    7.3029   9.0863                  11
 PROMEDIO Elastic Net    7.4192   8.1950                  10
 PROMEDIO       LASSO    7.4192   8.1950                  10

[ÉXITO] Archivo de datos históricos consolidado en Excel:
C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\6_recomendacion_estadistica_2\3_resultados\desempeno_regularizacion_2023-2025.xlsx
